# SOAR Project: ML Triage Analyzer (Cross-Dataset Testing)
This notebook trains the XGBoost Triage Analyzer. 
It generates 15,000 synthetic Wazuh alerts with realistic noise/overlap for **training**, and fetches 30,000 real KDD Cup '99 records (also with mapping noise) for **testing**.

In [ ]:
!pip install xgboost scikit-learn pandas numpy

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_kddcup99
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

### 1. Synthetic Data Generator (Dataset A - Train Set)

In [ ]:
def generate_synthetic_data(n_samples=15000):
    print("[*] Generating Synthetic Wazuh Dataset (Training set)...")
    np.random.seed(42)
    data = []
    
    attack_types = {
        "brute_force": {"rule_level": (10, 14), "failed_logins": (20, 150), "has_mitre_tag": 1},
        "ransomware": {"rule_level": (13, 15), "failed_logins": (0, 3), "has_mitre_tag": 1},
        "c2_beacon": {"rule_level": (8, 12), "failed_logins": (0, 1), "has_mitre_tag": 1},
        "sql_injection": {"rule_level": (9, 13), "failed_logins": (0, 2), "has_mitre_tag": 1}
    }
    
    for _ in range(n_samples // 2):
        # Malicious
        for atk, params in attack_types.items():
            real_rl = np.random.randint(params["rule_level"][0], params["rule_level"][1]+1)
            real_has_mitre = params["has_mitre_tag"]
            if np.random.rand() < 0.15: real_rl -= np.random.randint(3, 7)
            if np.random.rand() < 0.2: real_has_mitre = 0
            data.append({
                "rule_level": max(1, real_rl),
                "hour_of_day": np.random.randint(0, 24),
                "day_of_week": np.random.randint(0, 7),
                "failed_logins": np.random.randint(params["failed_logins"][0], params["failed_logins"][1]+1) if np.random.rand() > 0.1 else 0,
                "src_ip_is_internal": np.random.choice([0,1]),
                "src_ip_reputation": np.random.randint(50, 100),
                "agent_os": np.random.randint(0, 3),
                "event_count_1h": np.random.randint(10, 200),
                "is_fim_event": np.random.choice([0,1]) if np.random.rand() > 0.15 else 0,
                "has_mitre_tag": real_has_mitre,
                "label": 1
            })
        # Benign
        real_rl_b = np.random.randint(1, 6)
        if np.random.rand() < 0.1: real_rl_b += np.random.randint(3, 8)
        data.append({
                "rule_level": min(15, real_rl_b),
                "hour_of_day": np.random.randint(7, 19),
                "day_of_week": np.random.randint(0, 5),
                "failed_logins": np.random.randint(0, 2) if np.random.rand() > 0.05 else np.random.randint(3, 8),
                "src_ip_is_internal": 1,
                "src_ip_reputation": np.random.randint(0, 10),
                "agent_os": np.random.randint(0, 3),
                "event_count_1h": np.random.randint(1, 40),
                "is_fim_event": 0 if np.random.rand() > 0.05 else 1,
                "has_mitre_tag": 0,
                "label": 0
        })
    
    df = pd.DataFrame(data).sample(n_samples, replace=True).reset_index(drop=True)
    return df

### 2. Real Data Fetch & Map (Dataset B - Test Set)

In [ ]:
def fetch_and_map_real_data(n_samples=30000):
    print("[*] Downloading KDD Cup 99 Data (Real Testing Set)...")
    kdd = fetch_kddcup99(percent10=True, as_frame=True)
    df_raw = kdd.frame.sample(n=n_samples, random_state=42).reset_index(drop=True)
    
    print("[*] Mapping Network PCAP features to SIEM Wazuh Features...")
    mapped = []
    for idx, row in df_raw.iterrows():
        is_attack = row['labels'] != b'normal.'
        label = 1 if is_attack else 0
        
        if label == 1:
            base_rule = np.random.randint(9, 14)
            if row['hot'] > 0 or row['num_compromised'] > 0: base_rule = 14
            if np.random.rand() < 0.15: base_rule -= np.random.randint(4, 7)
            rule_level = max(1, base_rule)
        else:
            base_rule = np.random.randint(1, 6)
            if np.random.rand() < 0.1: base_rule += np.random.randint(3, 8)
            rule_level = min(15, base_rule)
        
        has_mitre = 1 if np.random.rand() > 0.25 else 0 if label == 1 else 1 if np.random.rand() < 0.05 else 0
            
        mapped.append({
            "rule_level": rule_level,
            "hour_of_day": np.random.randint(0, 6) if label == 1 else np.random.randint(8, 18),
            "day_of_week": np.random.randint(0, 7),
            "failed_logins": int(row['num_failed_logins']),
            "src_ip_is_internal": 0 if b'smurf' in row['labels'] else np.random.choice([0,1]),
            "src_ip_reputation": np.random.randint(50, 100) if label == 1 else np.random.randint(0, 10),
            "agent_os": np.random.randint(0, 3),
            "event_count_1h": max(int(row['count']) * 10, 1),
            "is_fim_event": 1 if int(row['num_file_creations']) > 0 else 0,
            "has_mitre_tag": has_mitre,
            "label": label
        })
    return pd.DataFrame(mapped)

### 3. Cross-Dataset Evaluation Pipeline

In [ ]:
df_train = generate_synthetic_data(15000)
df_test = fetch_and_map_real_data(30000)

features = [
    "rule_level", "hour_of_day", "day_of_week",
    "failed_logins", "src_ip_is_internal", "src_ip_reputation",
    "agent_os", "event_count_1h", "is_fim_event", "has_mitre_tag"
]

X_train, y_train = df_train[features], df_train['label']
X_test, y_test = df_test[features], df_test['label']

print(f"\n[*] Training Model STRICTLY on {len(df_train)} Synthetic Samples.")
print(f"[*] Testing Model STRICTLY on {len(df_test)} Real Samples.")

model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n[+] === CROSS-DATASET EVALUATION MATRICES ===")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))
print(f"Accuracy Score:  {accuracy_score(y_test, y_pred) * 100:.2f}%")
